# ==============================================================================
# 🫀 CARDIOLOGY SCREENING: HEART DISEASE RISK VIA ADABOOST CLASSIFIER
# ==============================================================================
### Core Principles:
Cardiovascular screening requires high diagnostic recall to prevent undetected coronary events.
AdaBoost constructs a sequential committee of decision stumps, iteratively increasing attention on subtle clinical indicator combinations.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, recall_score, roc_auc_score, accuracy_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Cardiology Environment ready.")


✅ STEP 1: Cardiology Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Cleveland Heart Disease dataset (1,025 clinical examinations).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/heart_disease/heart.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} patients, {df.shape[1]} physiological attributes")
print(f"Heart Disease Prevalence:\n{df['target'].value_counts(normalize=True).round(4)*100}%")
df.head(3)


📊 Dataset Shape: 920 patients, 14 physiological attributes
Heart Disease Prevalence:
target
1    55.33
0    44.67
Name: proportion, dtype: float64%


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `target` (1 = Heart Disease Present, 0 = Healthy).


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['target'])
y = df['target']

print(f"Diagnostic Features: {X.shape[1]}")
print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")


Diagnostic Features: 13
Missing attributes:
trestbps     59
chol         30
fbs          90
restecg       2
thalach      55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
dtype: int64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Patients: {X_train.shape[0]} ({y_train.sum()} Positive)")
print(f"Test Patients : {X_test.shape[0]} ({y_test.sum()} Positive)")


Train Patients: 736 (407 Positive)
Test Patients : 184 (102 Positive)


## ⚙️ STEP 7: CLINICAL COLUMN TRANSFORMER PREPROCESSOR
Imputing medical vitals via median and encoding categorical cardiology indicators.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Ensure strings for categorical handling
X_train = X_train.copy()
X_test = X_test.copy()
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Clinical Preprocessor configured.")


✅ STEP 7: Clinical Preprocessor configured.


## ⚠️ STEP 8: STUMP DEPTH DUEL (Depth 1 vs Depth 2 Base Trees)
Contrasting 1-split stumps against 2-split trees in AdaBoost.


In [6]:
# STEP 8: BASE TREE DEPTH DUEL
ada_depth1 = Pipeline([
    ('prep', preprocessor),
    ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=100, random_state=42))
])
ada_depth1.fit(X_train, y_train)

ada_depth2 = Pipeline([
    ('prep', preprocessor),
    ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2), n_estimators=100, random_state=42))
])
ada_depth2.fit(X_train, y_train)

rec1 = recall_score(y_test, ada_depth1.predict(X_test))
rec2 = recall_score(y_test, ada_depth2.predict(X_test))

print("="*65)
print(f"AdaBoost (Depth 1 Stumps) Recall: {rec1 * 100:.2f}% | Acc: {accuracy_score(y_test, ada_depth1.predict(X_test))*100:.2f}%")
print(f"AdaBoost (Depth 2 Trees)  Recall: {rec2 * 100:.2f}% | Acc: {accuracy_score(y_test, ada_depth2.predict(X_test))*100:.2f}%")
print("="*65)


AdaBoost (Depth 1 Stumps) Recall: 85.29% | Acc: 83.15%
AdaBoost (Depth 2 Trees)  Recall: 89.22% | Acc: 84.78%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `n_estimators` and `learning_rate` for medical recall.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'ada__n_estimators': [50, 100, 150],
    'ada__learning_rate': [0.05, 0.1, 0.5, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(ada_depth1, param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Cardiology Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Recall: {grid.best_score_ * 100:.2f}%")


🏆 Best Cardiology Hyperparameters: {'ada__learning_rate': 0.5, 'ada__n_estimators': 150}
🎯 Best 5-Fold CV Recall: 83.78%


## 📊 STEP 10 & 11: CARDIOLOGY REPORT & CONFUSION MATRIX
Reviewing diagnostic safety, false negative rates, and ROC-AUC.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("📋 CLINICAL CARDIAC SCREENING REPORT:")
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Heart Disease']))

print(f"🎯 Test Accuracy   : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Cardiac Recall  : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 ROC-AUC Score   : {roc_auc_score(y_test, y_prob)*100:.2f}%")

tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix:\n{cm}")
print(f"False Negatives (Missed Cardiac Conditions): {fn}")


📋 CLINICAL CARDIAC SCREENING REPORT:
               precision    recall  f1-score   support

      Healthy       0.87      0.82      0.84        82
Heart Disease       0.86      0.90      0.88       102

     accuracy                           0.86       184
    macro avg       0.86      0.86      0.86       184
 weighted avg       0.86      0.86      0.86       184

🎯 Test Accuracy   : 86.41%
🎯 Cardiac Recall  : 90.20%
🎯 ROC-AUC Score   : 91.10%

Confusion Matrix:
[[67 15]
 [10 92]]
False Negatives (Missed Cardiac Conditions): 10


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & EMERGENCY TRIAGE SMOKE TEST
Deploying pipeline and testing emergency triage screening latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/heart_disease_adaboost_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_risk = loaded_pipe.predict(sample_patient)[0]
pred_prob = loaded_pipe.predict_proba(sample_patient)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🫀 LIVE EMERGENCY ROOM CARDIAC TRIAGE:")
print(f"   Diagnostic Screening : {'🚨 CORONARY RISK DETECTED' if pred_risk == 1 else '🟢 NORMAL CARDIAC PROFILE'}")
print(f"   Condition Probability: {pred_prob*100:.2f}%")
print(f"   Screening Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/heart_disease_adaboost_pipeline.joblib (20,836 bytes)

🫀 LIVE EMERGENCY ROOM CARDIAC TRIAGE:
   Diagnostic Screening : 🟢 NORMAL CARDIAC PROFILE
   Condition Probability: 48.79%
   Screening Latency    : 40.457 ms (SLA < 2.0ms: CHECK)
